# GLProtein (Refactor) + TMVecLoss

This notebook sets up the environment, unzips the uploaded repo zip, installs dependencies (including TM-Vec)
and provides commands for verification and full pre-training.

In [1]:
# Mount Google Drive
from google.colab import drive, files
import os

drive.mount('/content/drive')


# Required files in your drive:
# (1) Repo zip: GLProtein_TMVec.zip
# (2) TM-Vec checkpoint (.ckpt)  (only needed if --use_tmvec_loss)
# (3) TM-Vec params (.json)  (only needed if --use_tmvec_loss)
# (4) UniProt data: uniprot_sprot.dat (or uniprot_sprot.dat.gz)
# (5) TM-Vec pairs TSV: tmvec_pairs.tsv (only needed if --use_tmvec_loss)
# (6) TM-Vec pair embeddings NPY: tmvec_pairs_emb.npy (only needed if --use_tmvec_loss)
REPO_ZIP_PATH     = "/content/drive/MyDrive/GLProtein_TMVec/GLProtein_TMVec.zip"
TMVEC_CKPT_PATH   = "/content/drive/MyDrive/GLProtein_TMVec/tm_vec_cath_model.ckpt"
TMVEC_JSON_PATH   = "/content/drive/MyDrive/GLProtein_TMVec/tm_vec_cath_model_params.json"
UNIPROT_DAT_PATH  = "/content/drive/MyDrive/GLProtein_TMVec/uniprot_sprot.dat.gz"
TMVEC_TSV_PATH    = "/content/drive/MyDrive/GLProtein_TMVec/tmvec_pairs.tsv"
TMVEC_NPY_PATH    = "/content/drive/MyDrive/GLProtein_TMVec/tmvec_pairs_emb.npy"

# Locations for assets inside the repo after unzip
TMVEC_ASSETS_DIR  = "assets/tmvec"
PRETRAIN_DATA_DIR = "data/pretrain_data"

def _exists(p: str) -> bool:
    return p and os.path.exists(p)

print("REPO_ZIP_PATH exists:", _exists(REPO_ZIP_PATH), REPO_ZIP_PATH)
print("TMVEC_CKPT_PATH exists:", _exists(TMVEC_CKPT_PATH), TMVEC_CKPT_PATH)
print("TMVEC_JSON_PATH exists:", _exists(TMVEC_JSON_PATH), TMVEC_JSON_PATH)
print("UNIPROT_DAT_PATH exists:", _exists(UNIPROT_DAT_PATH), UNIPROT_DAT_PATH)
print("TMVEC_TSV_PATH exists:", _exists(TMVEC_TSV_PATH), TMVEC_TSV_PATH)
print("TMVEC_NPY_PATH exists:", _exists(TMVEC_NPY_PATH), TMVEC_NPY_PATH)

if not _exists(REPO_ZIP_PATH):
    print("Please upload the repo zip")
    uploaded = files.upload()
    print("Uploaded:", list(uploaded.keys()))
    for k in uploaded.keys():
        if k.endswith(".zip"):
            REPO_ZIP_PATH = os.path.abspath(k)
            break
    print("Current repo zip path:", REPO_ZIP_PATH)

Mounted at /content/drive
REPO_ZIP_PATH exists: True /content/drive/MyDrive/GLProtein_TMVec/GLProtein_TMVec.zip
TMVEC_CKPT_PATH exists: True /content/drive/MyDrive/GLProtein_TMVec/tm_vec_cath_model.ckpt
TMVEC_JSON_PATH exists: True /content/drive/MyDrive/GLProtein_TMVec/tm_vec_cath_model_params.json
UNIPROT_DAT_PATH exists: True /content/drive/MyDrive/GLProtein_TMVec/uniprot_sprot.dat.gz
TMVEC_TSV_PATH exists: True /content/drive/MyDrive/GLProtein_TMVec/tmvec_pairs.tsv
TMVEC_NPY_PATH exists: True /content/drive/MyDrive/GLProtein_TMVec/tmvec_pairs_emb.npy


In [2]:
import os, zipfile, glob, shutil

assert 'REPO_ZIP_PATH' in globals(), "Repo zip path is undefined"

local_zip = "/content/repo.zip"
if os.path.abspath(REPO_ZIP_PATH) != local_zip:
    shutil.copy(REPO_ZIP_PATH, local_zip)
else:
    local_zip = REPO_ZIP_PATH

if os.path.exists("repo"):
    shutil.rmtree("repo")
os.makedirs("repo", exist_ok=True)

with zipfile.ZipFile(local_zip, "r") as z:
    z.extractall("repo")

root_candidates = []
for root, dirs, files_ in os.walk("repo"):
    if "run_pretrain_refactor.py" in files_:
        root_candidates.append(root)
assert len(root_candidates) > 0, "No run_pretrain_refactor.py after unzip"

repo_root = root_candidates[0]
print("Repo root:", repo_root)
os.chdir(repo_root)
print("Working directory:", os.getcwd())

Repo root: repo/GLProtein_TMVec
Working directory: /content/repo/GLProtein_TMVec


In [3]:
# Install dependencies
!pip -q install --upgrade pip
!pip -q install -r requirements.txt

# Install TM-Vec
!pip -q install git+https://github.com/tymor22/tm-vec.git

import tm_vec
print("Successfully imported tm_vec")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 32.8 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
Successfully imported tm_vec


In [4]:
import os, shutil, gzip

assert 'TMVEC_ASSETS_DIR' in globals(), "TM-Vec assets directory not found"
assert 'PRETRAIN_DATA_DIR' in globals(), "Data directory not found"

os.makedirs(TMVEC_ASSETS_DIR, exist_ok=True)
os.makedirs(PRETRAIN_DATA_DIR, exist_ok=True)

def copy_file(src_path: str, dst_path: str):
    os.makedirs(os.path.dirname(dst_path), exist_ok=True)
    shutil.copy(src_path, dst_path)

def copy_or_decompress_gz(src_path: str, dst_dir: str):
    """Copy a file into dst_dir. If it's a .gz, decompress into dst_dir."""
    if not src_path or not os.path.exists(src_path):
        return None

    lower = src_path.lower()
    os.makedirs(dst_dir, exist_ok=True)

    if lower.endswith(".gz") and not (lower.endswith(".tar.gz") or lower.endswith(".tgz")):
        out_path = os.path.join(dst_dir, os.path.basename(src_path[:-3]))
        with gzip.open(src_path, "rb") as fin, open(out_path, "wb") as fout:
            shutil.copyfileobj(fin, fout)
        return out_path

    out_path = os.path.join(dst_dir, os.path.basename(src_path))
    shutil.copy(src_path, out_path)
    return out_path

dat_dst = copy_or_decompress_gz(UNIPROT_DAT_PATH, PRETRAIN_DATA_DIR)
tsv_dst, npy_dst, ckpt_dst, json_dst = None, None, None, None
if TMVEC_TSV_PATH and os.path.exists(TMVEC_TSV_PATH):
    tsv_dst = copy_or_decompress_gz(TMVEC_TSV_PATH, PRETRAIN_DATA_DIR)
if TMVEC_NPY_PATH and os.path.exists(TMVEC_NPY_PATH):
    npy_dst = copy_or_decompress_gz(TMVEC_NPY_PATH, PRETRAIN_DATA_DIR)
if TMVEC_CKPT_PATH and os.path.exists(TMVEC_CKPT_PATH):
    ckpt_dst = copy_or_decompress_gz(TMVEC_CKPT_PATH, TMVEC_ASSETS_DIR)
if TMVEC_JSON_PATH and os.path.exists(TMVEC_JSON_PATH):
    json_dst = copy_or_decompress_gz(TMVEC_JSON_PATH, TMVEC_ASSETS_DIR)

print("assets/tmvec:", os.listdir(TMVEC_ASSETS_DIR))
print("data/pretrain_data:", os.listdir(PRETRAIN_DATA_DIR))

print("Destinations:")
print(".dat:", dat_dst)
print(".tsv:", tsv_dst)
print(".npy:", npy_dst)
print(".ckpt:", ckpt_dst)
print(".json:", json_dst)

if dat_dst is None:
    print("UniProt .dat/.dat.gz not copied. ProteinSeqDataset will fail.")
if tsv_dst is None:
    print("TM-Vec data TSV not copied. TMVecLoss will fail.")
if npy_dst is None:
    print("TM-Vec data NPY not copied. TMVecLoss will load an extra model during training.")
if ckpt_dst is None:
    print("TM-Vec checkpoint not copied. TMVecLoss will fail.")
if json_dst is None:
    print("TM-Vec params JSON not copied. TMVecLoss will fail.")



assets/tmvec: ['tm_vec_cath_model_params.json', 'tm_vec_cath_model.ckpt']
data/pretrain_data: ['.DS_Store', 'tmvec_pairs_emb.npy', 'uniprot_sprot.dat', 'tmvec_pairs.tsv']
Destinations:
.dat: data/pretrain_data/uniprot_sprot.dat
.tsv: data/pretrain_data/tmvec_pairs.tsv
.npy: data/pretrain_data/tmvec_pairs_emb.npy
.ckpt: assets/tmvec/tm_vec_cath_model.ckpt
.json: assets/tmvec/tm_vec_cath_model_params.json


## (Optional, only run if TSV/NPY is not present) TSV/NPY Generation for TM-Vec
For full pretraining, refer to (8) Full Pretraining in `SETUP_TMVecLoss.md` for list of arguments.

In [5]:
!pip -q install faiss-cpu
import faiss

In [6]:
!python generate_tmvec_pairs_tsv.py \
  --uniprot_dat data/pretrain_data/uniprot_sprot.dat \
  --out_tsv data/pretrain_data/tmvec_pairs.tsv \
  --out_emb_npy data/pretrain_data/tmvec_pairs_emb.npy \
  --tmvec_ckpt assets/tmvec/tm_vec_cath_model.ckpt \
  --tmvec_config assets/tmvec/tm_vec_cath_model_params.json \
  --device cuda \
  --max_proteins 500 \
  --pairs_per_anchor 1 \
  --top_k 5 \
  --emb_dtype float16

Loaded 500 sequences.
Lightning automatically upgraded your loaded checkpoint from v1.5.8 to v2.6.1. To apply the upgrade to your files permanently, run `python -m pytorch_lightning.utilities.upgrade_checkpoint assets/tmvec/tm_vec_cath_model.ckpt`
config.json: 100% 546/546 [00:00<00:00, 2.35MB/s]
pytorch_model.bin: 100% 11.3G/11.3G [01:26<00:00, 130MB/s]
model.safetensors:  84% 9.52G/11.3G [01:17<00:24, 70.4MB/s]
Loading weights:   0% 0/196 [00:00<?, ?it/s]
Loading weights:   1% 1/196 [00:00<00:00, 9554.22it/s, Materializing param=encoder.block.0.layer.0.SelfAttention.k.weight]
Loading weights:   1% 1/196 [00:00<00:00, 4804.47it/s, Materializing param=encoder.block.0.layer.0.SelfAttention.k.weight]
Loading weights:   1% 2/196 [00:00<00:00, 4032.98it/s, Materializing param=encoder.block.0.layer.0.SelfAttention.o.weight]
Loading weights:   1% 2/196 [00:00<00:00, 3355.44it/s, Materializing param=encoder.block.0.layer.0.SelfAttention.o.weight]
Loading weights:   2% 3/196 [00:00<00:00, 3679

## MLM-only verification run


In [8]:
# MLM-only quick verification
!python run_pretrain_refactor.py \
  --output_dir outputs/mlm_only_quick \
  --max_steps 10 \
  --per_device_train_batch_size 4 \
  --protein_seq_sample_limit 5

Process rank: -1, device: cpu, n_gpu: 1, distributed training: False, 16-bits training: False
Loading weights: 100% 487/487 [00:00<00:00, 813.49it/s, Materializing param=pooler.dense.weight]
BertModel LOAD REPORT from: Rostlab/prot_bert
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architectur

## TMVecLoss verification (global structure)

Requires:
- `data/pretrain_data/tmvec_pairs.tsv`
- `data/pretrain_data/tmvec_pairs_embed.npy` (prevents extra model loading)
- `assets/tmvec/tm_vec_cath_model.ckpt`
- `assets/tmvec/tm_vec_cath_model_params.json`

For full pretraining, refer to (8) Full Pretraining in `SETUP_TMVecLoss.md` for list of arguments.


In [5]:
!python run_pretrain_refactor.py \
  --output_dir outputs/mlm_plus_tmvec_precomputed \
  --use_tmvec_loss True \
  --tmvec_pairs_tsv tmvec_pairs.tsv \
  --tmvec_pairs_emb_npy tmvec_pairs_emb.npy \
  --max_steps 10 \
  --per_device_train_batch_size 4 \
  --fp16 \
  --protein_seq_sample_limit 5

Process rank: -1, device: cpu, n_gpu: 1, distributed training: False, 16-bits training: True
tokenizer_config.json: 100% 86.0/86.0 [00:00<00:00, 509kB/s]
vocab.txt: 100% 81.0/81.0 [00:00<00:00, 242kB/s]
special_tokens_map.json: 100% 112/112 [00:00<00:00, 813kB/s]
config.json: 100% 361/361 [00:00<00:00, 2.09MB/s]
pytorch_model.bin: 100% 1.68G/1.68G [00:13<00:00, 129MB/s]
model.safetensors:   0% 0.00/1.68G [00:00<?, ?B/s]
Loading weights:   0% 0/487 [00:00<?, ?it/s]
Loading weights:   0% 1/487 [00:00<00:00, 9709.04it/s, Materializing param=embeddings.LayerNorm.bias]
Loading weights:   0% 1/487 [00:00<00:00, 4826.59it/s, Materializing param=embeddings.LayerNorm.bias]
Loading weights:   0% 2/487 [00:00<00:00, 4578.93it/s, Materializing param=embeddings.LayerNorm.weight]
Loading weights:   0% 2/487 [00:00<00:00, 3708.49it/s, Materializing param=embeddings.LayerNorm.weight]
Loading weights:   1% 3/487 [00:00<00:00, 4094.67it/s, Materializing param=embeddings.position_embeddings.weight]
Loadi